In [1]:
# warning 무시
import warnings
warnings.filterwarnings("ignore")

import tensorflow as tf

# 새로운 초기 가중치로 처음부터 다시 학습
tf.keras.backend.clear_session()

# TensorFlow가 GPU 메모리를 처음부터 크게 선점하지 않고, 필요한 만큼 점진적으로 사용하도록 설정
gpus = tf.config.list_physical_devices("GPU")

for gpu in gpus:
    tf.config.experimental.set_memory_growth(gpu, True)

2026-08-31 06:18:49.377629: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-31 06:18:49.402498: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-08-31 06:18:49.424509: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-08-31 06:18:49.431207: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-08-31 06:18:49.446572: I tensorflow/core/platform/cpu_feature_guar

In [2]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.resnet50 import decode_predictions
import numpy as np


In [4]:
model = ResNet50(weights='imagenet') # 첫 실행 시 가중치 다운로드됨
 
# https://commons.wikimedia.org/wiki/File:YellowLabradorLooking_new.jpg
img = image.load_img('./images/YellowLabradorLooking_new.jpg',target_size=(224, 224))
x = image.img_to_array(img)    # x.shape=(224,224,3)
x = np.expand_dims(x, axis=0)  # x.shape=(1,244,244,3)
pred = model.predict(x, verbose=0)
print('Predicted:', decode_predictions(pred, top=3))


I0000 00:00:1788124777.246541   98070 service.cc:146] XLA service 0x77ed14001f80 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1788124777.247087   98070 service.cc:154]   StreamExecutor device (0): NVIDIA GeForce GTX 1050, Compute Capability 6.1
2026-08-31 06:19:38.235678: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
2026-08-31 06:19:38.742512: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:531] Loaded cuDNN version 8907


I0000 00:00:1788124782.319883   98070 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


35363/35363 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Predicted: [[('n02099712', 'Labrador_retriever', 0.2689771), ('n02108089', 'boxer', 0.15149526), ('n02099849', 'Chesapeake_Bay_retriever', 0.10756575)]]


# 전이학습 + 미세조정

In [ ]:
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
resnet_model = ResNet50(input_shape=(224,224,3),include_top=False)

# 전이학습 + 전체 미세조정(Full Fine-tuning).
resnet_model.trainable = True
model = Sequential()
model.add(resnet_model)
model.add(Flatten())
model.add(Dense(1024, activation='relu')) # FC 층 추가
model.add(Dense(3, activation='softmax')) # 3개 클래스 분류
model.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ resnet50 (Functional)                │ (None, 7, 7, 2048)          │      23,587,712 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_2 (Flatten)                  │ (None, 100352)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 1024)                │     102,761,472 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 3)                   │           3,075 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 126,352,259 (482.00 MB)

 Trainable params: 126,299,139 (481.79 MB)

 Non-trainable params: 53,120 (207.50 KB)